# Interactive Peak Fitting Tool for FIB TOF-SIMS / Mass Spectra

## Overview

This Jupyter Notebook-based tool is designed for interactive peak fitting of FIB TOF-SIMS and other mass spectrometry data. It supports Gaussian, Lorentzian, and PseudoVoigt models with optional baseline correction, smoothing, and default settings on uranium isotope ratio analysis.

**Current Version:** 1.6.4 **(234U included, particle region only)**

**Author:** Xiao Sun [https://github.com/xiaosun622]

---

## Workflow: Step-by-Step Procedure

### 1. Load Spectrum File

* Reads a tab-delimited `.txt` file.
* Expected columns:

  * `mass/charge (m/Q)` (x-axis)
  * `Total (cts/TOF-Extraction)` (intensity)

### 2. Apply Smoothing (Optional)

* Savitzky–Golay smoothing filter reduces noise while preserving peak shape.
* Adjustable parameters:

  * `Smooth Win`: Window size
  * `Polyorder`: Polynomial order used in smoothing

### 3. Define Peak Regions

* User inputs peak center and range width (±) for each ion.
* Regions are defined as: `center ± range_width`

### 4. Baseline Correction (if enabled)

* Options:

  * `average`: Flat baseline using predicted intensity ± offset
  * `linear`: Line fit to surrounding regions
  * `polynomial`: 2nd-order polynomial fit
* This background is subtracted from the signal to isolate the peak.

### 5. Select Fit Model

* Choose between symmetric or asymmetric peak shapes:

  * Gaussian
  * Lorentzian
  * PseudoVoigt (or Voigt for asymmetric cases)

### 6. Fit the Peak

* The fitting is applied to the **baseline-corrected signal**.
* Initial parameters are estimated (center, amplitude, sigma).
* The model is optimized to minimize the squared difference between corrected data and prediction.

### 7. Extract Results

* From the fit result:

  * Best-fit curve (`model_prediction`)
  * Fitting Deviation (`data - prediction`)
  * R² (goodness of fit)
  * Area (peak amplitude)

### 8. Plot Outputs

Each subplot includes:

* Smoothed signal
* Corrected signal
* Model fit
* **Fitting Deviation** (formerly “Residuals”)

### 9. Calculate Isotope Ratios

For isotopes (e.g. 234U, 235U, 238U), the ratio is:

```math
\text{Ratio} = \frac{\text{Area}_{235}}{\text{Area}_{234} + \text{Area}_{235} + \text{Area}_{238}}
```

### 10. Save Outputs (Manual Trigger)

* Press "Save Results" after fitting to export:

  * A PNG image of all plots
  * A CSV summary table with areas, R², and isotope ratios

---

## Terminology

| Term                  | Meaning                                                         |
| --------------------- | --------------------------------------------------------------- |
| **Fitting Deviation** | Difference between corrected data and model fit                 |
| **Baseline**          | Estimated background under the peak (subtracted before fitting) |
| **Best Fit**          | Model prediction using optimized parameters                     |
| **R²**                | Coefficient of determination; closer to 1 means better fit      |

---

## Requirements

* Python 3.7+
* `pandas`, `numpy`, `matplotlib`, `scipy`, `lmfit`, `ipywidgets`

Install via pip:

```bash
pip install pandas numpy matplotlib scipy lmfit ipywidgets
```

---

MIT License

Copyright (c) 2025 xiaosun622

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [1]:
%pip install pandas numpy matplotlib scipy lmfit ipywidgets

In [3]:
# === Interactive Peak Fitting Script for TOF-SIMS / Mass Spectra ===
"""
Title: Interactive Peak Fitting Tool
Author: Xiao Sun [https://github.com/xiaosun622]
Version: 1.6.4 ((234U included, particle region only))
Date: 11-12-2025

Description:
This script provides an interactive Jupyter Notebook-based interface for peak fitting in TOF-SIMS or general mass spectrometry spectra.
It enables users to upload a tab-delimited spectrum file, smooth data, select fit models, perform baseline correction, and calculate isotope ratios.
"""

# === [IMPORTS] ===
# Core libraries for numerical work, data handling, and plotting
import os, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Signal processing (Savitzky–Golay smoothing)
from scipy.signal import savgol_filter

# Peak-shape models for non-linear least-squares fitting
from lmfit.models import GaussianModel, LorentzianModel, PseudoVoigtModel, VoigtModel

# Jupyter interactive UI
import ipywidgets as widgets
from IPython.display import display, clear_output

# Try importing skewed Gaussian model; fall back gracefully if missing
try:
    from lmfit.models import SkewedGaussianModel
except ImportError:
    SkewedGaussianModel = None

# Suppress non-critical warnings from the uncertainties package
warnings.filterwarnings("ignore", category=UserWarning, module="uncertainties.core")

# === [DATA INIT] ===
# Default column names expected from IONTOF TXT export
x_column = 'mass/charge (m/Q)'
y_column = 'Total (cts/TOF-Extraction)'  # kept for reference; Y is actually taken from 5th column

# Global state containers (populated after upload and fitting)
data = None
x = None
y = None
fig = None
df_summary = None
results_last_filename = None

# === [WIDGETS] ===
# File upload widget (.txt spectrum)
upload_widget = widgets.FileUpload(
    accept='.txt',
    multiple=False,
    description='Upload .txt',
    layout=widgets.Layout(width='260px')
)

# Fitting options: peak model and symmetry
fit_model = widgets.Dropdown(
    options=['gaussian', 'lorentzian', 'pseudovoigt'],
    value='pseudovoigt',
    description='Fit Model:'
)
symmetric_fit = widgets.Checkbox(value=False, description='Symmetric Peak')

# Baseline correction controls
use_baseline = widgets.Checkbox(value=True, description='Baseline Correction')
baseline_type = widgets.Dropdown(
    options=['average', 'linear', 'polynomial'],
    value='linear',
    description='Baseline:'
)

# Smoothing parameters for Savitzky–Golay filter
smoothing_window = widgets.IntSlider(
    value=11, min=3, max=51, step=2,
    description='Smooth Win:',
    layout=widgets.Layout(width='320px')
)
smoothing_poly = widgets.IntSlider(
    value=3, min=1, max=5, step=1,
    description='Polyorder:',
    layout=widgets.Layout(width='260px')
)

# Acquisition parameters for converting Area → Counts
pixel_widget = widgets.IntText(
    value=256,
    description='Pixel:',
    layout=widgets.Layout(width='160px')
)
frames_widget = widgets.IntText(
    value=250,
    description='Frames:',
    layout=widgets.Layout(width='160px')
)

# Target isotopic peaks to fit
region_labels = [
    '234U', '235U', '238U',
    '234UO', '235UO', '238UO',
    '234UO2', '235UO2', '238UO2'
]

# Default mass centers for each species (m/z)
peak_centers = {
    '234U': 234.0, '235U': 235.0, '238U': 238.0,
    '234UO': 250.0, '235UO': 251.0, '238UO': 254.0,
    '234UO2': 266.0, '235UO2': 267.0, '238UO2': 270.0
}

# Per-region UI controls: center, ±range, ±baseline window
range_width_widgets = {}
baseline_offset_widgets = {}
for label in region_labels:
    # Center m/z of the region
    center_w = widgets.FloatText(
        value=peak_centers[label],
        description=f'{label}:',
        layout=widgets.Layout(width='170px')
    )
    # Default fit window: 234-series narrower (±0.5), others ±1.0
    default_width = 0.5 if label in ['234U', '234UO', '234UO2'] else 1.0
    width_w = widgets.FloatText(
        value=default_width,
        description='±range:',
        layout=widgets.Layout(width='150px')
    )
    # Baseline sampling window around each peak
    bsoff_w = widgets.FloatText(
        value=1.0,
        description='±baseline:',
        layout=widgets.Layout(width='170px')
    )
    range_width_widgets[label] = (center_w, width_w)
    baseline_offset_widgets[label] = bsoff_w

# Action buttons for running fits and saving results
fit_button = widgets.Button(
    description='Fit and Calculate',
    layout=widgets.Layout(width='220px')
)
save_button = widgets.Button(
    description='Save Results',
    layout=widgets.Layout(width='160px')
)

# Layout of the full user interface
ui = widgets.VBox([
    upload_widget,
    widgets.HBox([fit_model, symmetric_fit]),
    widgets.HBox([use_baseline]),
    widgets.HBox([baseline_type]),
    widgets.HBox([smoothing_window, smoothing_poly]),
    widgets.HBox([pixel_widget, frames_widget]),
    widgets.HTML('<b>Edit Peak Centers and Ranges</b>'),
    *[
        widgets.HBox([
            range_width_widgets[l][0],
            range_width_widgets[l][1],
            baseline_offset_widgets[l]
        ])
        for l in region_labels
    ],
    widgets.HBox([fit_button, save_button])
])

# === [MODEL SELECTION] ===
def get_model(model_type, symmetric=True):
    """
    Return an lmfit peak model based on selected model type and symmetry.

    Parameters
    ----------
    model_type : str
        'gaussian', 'lorentzian', or 'pseudovoigt'.
    symmetric : bool
        If False, attempt to use an asymmetric variant where available.

    Returns
    -------
    lmfit.Model
    """
    if model_type == 'gaussian':
        # Use SkewedGaussianModel if asymmetric fit requested and available
        return GaussianModel() if symmetric else (
            SkewedGaussianModel() if SkewedGaussianModel else GaussianModel()
        )
    if model_type == 'lorentzian':
        # Lorentzian or more general Voigt if asymmetric
        return LorentzianModel() if symmetric else VoigtModel()
    if model_type == 'pseudovoigt':
        # Pseudo-Voigt or Voigt fallback
        return PseudoVoigtModel() if symmetric else VoigtModel()
    raise ValueError('Unsupported model type')


# === [PEAK FITTING WITH BASELINE] ===
def fit_peak_custom_baseline(
    x_all, y_all, model_type, mz_range,
    baseline_offset, symmetric, use_bs, bs_mode
):
    """
    Fit a single peak region with optional baseline correction.

    Steps:
    1. Slice x_all, y_all to the selected m/z range.
    2. Perform an initial fit to estimate shape.
    3. Compute and subtract baseline (average, linear, or polynomial).
    4. Refit the baseline-corrected data.
    5. Return model fit, corrected curve, area (amplitude), and R².
    """
    # Restrict to target m/z interval
    mask = (x_all >= mz_range[0]) & (x_all <= mz_range[1])
    x_peak = x_all[mask]
    y_peak = y_all[mask]
    if x_peak.size < 5:
        raise ValueError('Peak range too narrow')

    # Initial model and parameter guesses
    model = get_model(model_type, symmetric)
    c_guess = (mz_range[0] + mz_range[1]) / 2
    h_guess = float(np.max(y_peak))
    s_guess = (mz_range[1] - mz_range[0]) / 4
    a_guess = h_guess * s_guess * np.sqrt(2 * np.pi)
    params = model.make_params(center=c_guess, amplitude=a_guess, sigma=s_guess)
    init_fit = model.fit(y_peak, params, x=x_peak)

    # Baseline correction (average, linear, polynomial, or none)
    if use_bs:
        if bs_mode == 'average':
            # Use model prediction at ±baseline_offset for flat baseline
            lval = np.interp(c_guess - baseline_offset, x_peak, init_fit.best_fit)
            rval = np.interp(c_guess + baseline_offset, x_peak, init_fit.best_fit)
            baseline = (lval + rval) / 2
            y_corr = y_peak - baseline

        elif bs_mode == 'linear':
            # Linear baseline estimated from sidebands
            c = c_guess
            lmask = (x_all >= c - baseline_offset) & (x_all < c - baseline_offset / 2)
            rmask = (x_all > c + baseline_offset / 2) & (x_all <= c + baseline_offset)
            xb = np.concatenate([x_all[lmask], x_all[rmask]])
            yb = np.concatenate([y_all[lmask], y_all[rmask]])
            p = np.polyfit(xb, yb, 1)
            y_corr = y_peak - np.polyval(p, x_peak)

        elif bs_mode == 'polynomial':
            # Quadratic baseline fit within peak window
            p = np.polyfit(x_peak, y_peak, 2)
            y_corr = y_peak - np.polyval(p, x_peak)
        else:
            y_corr = y_peak
    else:
        y_corr = y_peak

    # Final fit on baseline-corrected data
    final_fit = model.fit(
        y_corr,
        model.make_params(center=c_guess, amplitude=a_guess, sigma=s_guess),
        x=x_peak
    )

    # Calculate R² as a goodness-of-fit metric
    resid = y_corr - final_fit.best_fit
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y_corr - np.mean(y_corr)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    # For this script, "Area" uses the fitted amplitude parameter directly
    area = final_fit.params['amplitude'].value

    return final_fit, x_peak, y_peak, y_corr, area, resid, r2


# === [MAIN FITTING CALLBACK] ===
def run_fitting(_):
    """
    Callback for 'Fit and Calculate' button.

    Uses current global x, y arrays and UI settings to:
    - Smooth data
    - Fit each isotopic region
    - Plot all fits
    - Build result table with Area, Counts, R², and 235U ratios.
    """
    global data, x, y, fig, results_last_filename, df_summary

    clear_output(wait=True)
    display(ui)

    # Ensure data is loaded
    if x is None or y is None:
        print('Please upload a .txt file.')
        return

    # Smoothing window must be shorter than the spectrum
    if smoothing_window.value >= len(y):
        print("Error: Smoothing window too large.")
        return

    # Apply Savitzky–Golay smoothing to raw y data
    y_smooth = savgol_filter(y, smoothing_window.value, smoothing_poly.value)

    # Build per-region m/z windows and baseline offsets from widgets
    ranges = {
        l: (
            range_width_widgets[l][0].value - range_width_widgets[l][1].value,
            range_width_widgets[l][0].value + range_width_widgets[l][1].value
        )
        for l in region_labels
    }
    offsets = {l: baseline_offset_widgets[l].value for l in region_labels}

    # Prepare 3×3 subplot grid for the nine ion species
    fig, axes = plt.subplots(3, 3, figsize=(14, 14))
    axes = axes.flatten()
    areas = {}
    r2s = {}

    # Fit each defined region
    for i, l in enumerate(region_labels):
        try:
            res, xfit, yfit, ycorr, area, resid, r2 = fit_peak_custom_baseline(
                x, y_smooth, fit_model.value, ranges[l],
                offsets[l], symmetric_fit.value,
                use_baseline.value, baseline_type.value
            )
            areas[l] = area
            r2s[l] = r2

            ax = axes[i]
            ax.plot(xfit, yfit, label='Smoothed')
            ax.plot(xfit, res.best_fit, 'r--')
            ax.grid(True)
            ax.set_title(f"{l} (R²={r2:.2f})")
        except Exception as e:
            # Continue fitting other peaks even if one fails
            print(f'Error fitting {l}: {e}')
            continue

    plt.tight_layout()
    plt.show()

    # Build initial results table (Area and R² numeric; ratio column to be filled later)
    rows = []
    for l in region_labels:
        rows.append({
            'Ions': l,
            'Area': areas.get(l, np.nan),
            'R²': r2s.get(l, np.nan),
            '235U Isotopic Ratio': ''
        })
    df = pd.DataFrame(rows)

    # === COUNTS CALCULATION ===
    # Convert fitted Area to total Counts using acquisition parameters:
    # Counts = Area × Frames × Pixel²
    pixel = pixel_widget.value
    frames = frames_widget.value

    df.insert(1, 'Counts', df['Area'] * frames * (pixel ** 2))
    df['Counts'] = pd.to_numeric(df['Counts'], errors='coerce')

    # Force 234-series counts to be non-negative (no negative activity)
    df.loc[
        df['Ions'].isin(['234U', '234UO', '234UO2']) &
        (df['Counts'] <= 0),
        'Counts'
    ] = 0

    # === PER-PEAK 235U RATIOS (from Counts) ===
    # For each family (U, UO, UO2):
    # ratio_235 = Counts(235) / [Counts(234) + Counts(235) + Counts(238)]
    def calc_ratio_counts(c234, c235, c238):
        s = c234 + c235 + c238
        return c235 / s if s > 0 else np.nan

    for a234, a235, a238 in [
        ('234U', '235U', '238U'),
        ('234UO', '235UO', '238UO'),
        ('234UO2', '235UO2', '238UO2')
    ]:
        c234 = df.loc[df['Ions'] == a234, 'Counts'].sum(skipna=True)
        c235 = df.loc[df['Ions'] == a235, 'Counts'].sum(skipna=True)
        c238 = df.loc[df['Ions'] == a238, 'Counts'].sum(skipna=True)
        r = calc_ratio_counts(c234, c235, c238)
        if not np.isnan(r):
            df.loc[df['Ions'] == a235, '235U Isotopic Ratio'] = f"{r * 100:.2f}%"

    # === TOTAL U-SERIES SUMS AND OVERALL 235U RATIO ===
    def sum_counts_numeric(names):
        """Sum Counts (numeric) over a list of ion labels."""
        return df.loc[df['Ions'].isin(names), 'Counts'].sum(skipna=True)

    u234 = ['234U', '234UO', '234UO2']
    u235 = ['235U', '235UO', '235UO2']
    u238 = ['238U', '238UO', '238UO2']

    s234 = sum_counts_numeric(u234)
    s235 = sum_counts_numeric(u235)
    s238 = sum_counts_numeric(u238)

    # Global 235U fraction over all species
    ratio235_total = (s235 / (s234 + s235 + s238) * 100) if (s234 + s235 + s238) > 0 else np.nan

    # Append sum rows for U, UO, UO2 totals
    summary = pd.DataFrame([
        {
            'Ions': 'Sum(234U,234UO,234UO2)',
            'Area': np.nan,
            'Counts': s234,
            'R²': '',
            '235U Isotopic Ratio': ''
        },
        {
            'Ions': 'Sum(235U,235UO,235UO2)',
            'Area': np.nan,
            'Counts': s235,
            'R²': '',
            '235U Isotopic Ratio': f"{ratio235_total:.2f}%" if not np.isnan(ratio235_total) else ''
        },
        {
            'Ions': 'Sum(238U,238UO,238UO2)',
            'Area': np.nan,
            'Counts': s238,
            'R²': '',
            '235U Isotopic Ratio': ''
        }
    ])

    df = pd.concat([df, summary], ignore_index=True)

    # === FINAL FORMATTING FOR DISPLAY ===
    # Keep Area numeric (fit amplitude); format Counts and R² as strings.
    df['R²'] = pd.to_numeric(df['R²'], errors='coerce')
    df['Counts'] = df['Counts'].apply(
        lambda v: '' if pd.isna(v) else f"{v:.2f}"
    )
    df['R²'] = df['R²'].apply(
        lambda v: '' if pd.isna(v) else f"{v:.2f}"
    )

    # Set table index to 1..N for readability
    df.index = np.arange(1, len(df) + 1)
    display(df)

    # Store for saving
    df_summary = df.copy()
    if results_last_filename is None:
        results_last_filename = 'output'


# === [SAVE RESULTS] ===
def save_results(b):
    """
    Callback for 'Save Results' button.

    Saves:
    - PNG of the fit figure (if available)
    - CSV of the result table
    - Fitting settings and region definitions appended in the CSV.
    """
    global df_summary, fig, results_last_filename
    if df_summary is None:
        print("No results to save. Please run the fitting first.")
        return

    base_name = results_last_filename or 'output'

    # Save figure if generated
    try:
        if fig is not None:
            fig.savefig(f"{base_name}_fit.png", dpi=300)
    except Exception as e:
        print(f"Warning: could not save figure: {e}")

    # Save table plus metadata
    settings = {
        'Fitting Model': fit_model.value,
        'Symmetric Peak': symmetric_fit.value,
        'Baseline Correction': use_baseline.value,
        'Baseline Type': baseline_type.value,
        'Smoothing Window': smoothing_window.value,
        'Smoothing Polyorder': smoothing_poly.value,
        'Pixel': pixel_widget.value,
        'Frames': frames_widget.value
    }

    csv_path = f"{base_name}_summary.csv"
    with open(csv_path, 'w', encoding='utf-8') as f:
        df_summary.to_csv(f, index=False)
        f.write('\nFitting Parameters Used:\n')
        for k, v in settings.items():
            f.write(f" {k}: {v}\n")
        f.write('Peak Regions (m/z range):\n')
        for label in region_labels:
            c = range_width_widgets[label][0].value
            w = range_width_widgets[label][1].value
            o = baseline_offset_widgets[label].value
            f.write(
                f" {label}: center = {c}, ±range = {w}, ±baseline = {o}\n"
            )
    print(f"Saved: {base_name}_summary.csv and {base_name}_fit.png")


# === [FILE UPLOAD HANDLER] ===
def handle_upload(change):
    """
    Handle the uploaded .txt file:
    - Parse with pandas (tab or auto separator).
    - Select X and Y columns:
        * X: 'mass/charge (m/Q)' if present, else 3rd, else 1st column.
        * Y: always 5th column (cols[4]) if available.
      If fewer than 5 columns, fall back to first two numeric columns.
    - Store x, y as numpy arrays for fitting.
    """
    global data, x, y, results_last_filename
    uploaded = upload_widget.value
    if not uploaded:
        print('No file uploaded.')
        return

    # Extract uploaded file info from the FileUpload widget
    file_info = list(uploaded.values())[0] if isinstance(uploaded, dict) else uploaded[0]
    name = file_info.get('name', 'input.txt')
    content = file_info.get('content', b'')

    # Try parsing the file: first as tab-separated, then auto-detected separator
    df = None
    err = None
    for reader in (
        lambda: pd.read_csv(io.BytesIO(content), sep='\t', comment='#', engine='python'),
        lambda: pd.read_csv(io.BytesIO(content), sep=None, comment='#', engine='python')
    ):
        try:
            df = reader()
            break
        except Exception as e:
            err = str(e)

    if df is None:
        print(f'Failed to parse file: {err}')
        return

    # --- Select X and Y columns based on column order ---
    cols = df.columns.tolist()

    if len(cols) >= 5:
        # X: prefer named x_column if available, else 3rd, else 1st
        if x_column in df.columns:
            x_series = df[x_column]
        elif len(cols) >= 3:
            x_series = df[cols[2]]
        else:
            x_series = df[cols[0]]

        # Y: always take the 5th column, even if header differs between files
        y_series = df[cols[4]]
        print(f"Using X = '{x_series.name}', Y = '{y_series.name}' (5th column).")
    else:
        # Fallback: use first two numeric columns if fewer than 5 columns
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) >= 2:
            x_series, y_series = df[num_cols[0]], df[num_cols[1]]
            print(f"Warning: using {num_cols[0]} and {num_cols[1]}.")
        else:
            print('Numeric columns missing.')
            return

    # Drop NaNs and convert to numpy arrays
    valid = (~pd.isna(x_series)) & (~pd.isna(y_series))
    x_vals = np.asarray(x_series[valid].values, float)
    y_vals = np.asarray(y_series[valid].values, float)

    if x_vals.size < 10:
        print(f'Not enough points: {x_vals.size}.')
        return

    # Persist dataset and base filename for later save
    data = df
    x = x_vals
    y = y_vals
    results_last_filename = os.path.splitext(name)[0]

    clear_output(wait=True)
    display(ui)
    print(
        f"Loaded: {name} | {len(df)} rows | "
        f"Using columns: X='{x_series.name}', Y='{y_series.name}'"
    )


# === [WIRING CALLBACKS] ===
fit_button.on_click(run_fitting)
save_button.on_click(save_results)
upload_widget.observe(handle_upload, names='value')

# Display UI when the script is run in a Jupyter environment
display(ui)
